# Cell 1: download wheels

In [ ]:
!pip download --only-binary=:all: --no-deps -d /kaggle/working/wheels pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg pylibjpeg-rle python-gdcm
!ls -la /kaggle/working/wheels

# Cell 2: sample files in each format

In [ ]:
import os, shutil
from pydicom.data import get_testdata_file

os.makedirs("/kaggle/working/samples", exist_ok=True)
for name in ["JPEG-LL.dcm", "MR2_J2KR.dcm", "JPEG2000.dcm", "JPEG-lossy.dcm",
             "MR_small_RLE.dcm", "MR_small_implicit.dcm"]:
    try:
        shutil.copy(get_testdata_file(name), "/kaggle/working/samples/")
        print("ok  ", name)
    except Exception as err:
        print("fail", name, repr(err)[:100])

# Offline check

Run in a separate notebook with Internet off and this notebook's output attached, the same
way 05b-infer installs the decoders. Result: JPEG Lossless, JPEG 2000 (both variants),
RLE and Implicit VR all decode. JPEG Extended (12-bit lossy) fails, but the host does not
list it for the test set; 05b falls back to another series or the prior if it ever appears.

In [ ]:
import os, glob, subprocess, sys

def find_dir(name):
    for top in os.listdir("/kaggle/input"):
        if top == "competitions":
            continue
        hits = glob.glob(f"/kaggle/input/{top}/**/{name}", recursive=True)
        if hits:
            return hits[0]

wheels, samples = find_dir("wheels"), find_dir("samples")
print(wheels, samples)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--find-links", wheels,
                "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg", "pylibjpeg-rle",
                "python-gdcm"], check=True)

import pydicom
for p in sorted(glob.glob(f"{samples}/*.dcm")):
    ds = pydicom.dcmread(p)
    name = ds.file_meta.TransferSyntaxUID.name
    try:
        a = ds.pixel_array
        print(f"OK    {name:<45} {a.shape} {a.dtype}")
    except Exception as err:
        print(f"FAIL  {name:<45} {repr(err)[:120]}")